# 05 Attention U-Net Training

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys
sys.path.append('/content/unet-ensemble')

## Dataset Preparation

In [ ]:
!pip install safetensors huggingface_hub segmentation-models-pytorch

In [6]:
import os
import copy
import torch
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

from safetensors.torch import save_file
from torch.utils.data import DataLoader
from src.dataset.dataset import Dataset as LazyDS
from src.dataset.dataloader import DataLoader as DSLoader
from src.training.attention_unet import MBENAttentionUNet, Train
from src.utils.checkpoint_manager import CheckpointManager

In [ ]:
from huggingface_hub import login
token=''
login(token=token)

In [7]:
IMG_SIZE = 256
DATASET_ROOT = '/content/drive/Shareddrives/U-Net Ensemble/Dataset'
BATCH_SIZE  = 8
LR          = 1e-4
EPOCHS      = 50
PATIENCE    = 10           # early stopping patience
# --- Modified for Attention U-Net ---
CHECKPOINT = 'best_attention_unet.pth'
CHECKPOINT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/checkpoint/AttentionUNet_Checkpoints'
RESUME_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'resume_checkpoint.pth')
NUM_WORKERS = 2

# ── Feature combination to use ──────────────────────────────────────────────
# Choose ONE of the four supported combinations:
#   ('prnu', 'illumination', 'frequency')  — all three features
#   ('prnu', 'frequency')                  — PRNU + Frequency
#   ('prnu', 'illumination')               — PRNU + Illumination
#   ('frequency', 'illumination')          — Frequency + Illumination
FEATURES = ('prnu', 'illumination', 'frequency')

In [8]:
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)
    print(f"Created new checkpoint directory: {CHECKPOINT_DIR}")

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
loader = DSLoader(mask_folder       = "GTA - Masks",
                prnu_folder         = "Feature - PRNU",
                illumination_folder = "Feature - Illumination",
                frequency_folder    = "Feature - Frequency",
                categories          = ['Synthetic', 'Authentic'],
                templates           = ['template-a', 'template-albania', 'template-b',
                                        'template-c', 'template-chile', 'template-deutschland',
                                        'template-spain', 'template-usa'],
                features            = FEATURES)

In [ ]:
train_samples = loader.load_images('Training', DATASET_ROOT)
val_samples   = loader.load_images('Validation', DATASET_ROOT)

train_ds = LazyDS(train_samples, IMG_SIZE, augment=True,  features=FEATURES)
val_ds   = LazyDS(val_samples,   IMG_SIZE, augment=False, features=FEATURES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## MBEN + U-Net++ Model

Architecture overview:
```
PRNU  (1,H,W) ──► Branch-1 CNN ──┐
Illu  (1,H,W) ──► Branch-2 CNN ──┼──► Element-wise SUM  ──┐
Freq  (1,H,W) ──► Branch-3 CNN ──┘                         ├──► Concat ──► Projection Conv ──► U-Net++
Fused (3,H,W) ──► Concat Stem ─────────────────────────────┘
```
- **MBEN branches**: three independent CNN encoders let each feature learn its own representation before being summed.
- **Concat stem**: a shallow CNN on the stacked 3-channel input preserves inter-feature spatial relationships.
- **Projection**: merges both paths into 3 channels for the EfficientNet-B4 backbone of U-Net++.

In [ ]:
MBEN_OUT_CHANNELS = 64

model = MBENAttentionUNet(mben_out_ch=MBEN_OUT_CHANNELS, features=FEATURES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Active features : {sorted(FEATURES)}')
print(f'Trainable parameters: {total_params:,}')

## Loss, Optimizer, Scheduler

In [13]:
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

## Checkpoint Utilities

In [ ]:
train = Train(device, dice_loss, features=FEATURES)

checkpoint = CheckpointManager()

# ── Load checkpoint if one exists (resumes automatically) ──
(
    start_epoch,
    best_val_loss,
    early_stop_counter,
    train_losses,
    val_losses,
) = checkpoint.load_checkpoint(model, optimizer, scheduler, RESUME_CHECKPOINT)

best_model_wts = copy.deepcopy(model.state_dict())

## Training Loop

In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train.run_epoch(train_loader, model, optimizer, train=True)
    val_loss   = train.run_epoch(val_loader,   model, train=False)

    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch [{epoch:03d}/{EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  |  Val Loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss      = val_loss
        best_model_wts     = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
        save_file({k: v.cpu() for k, v in best_model_wts.items()}, f'{CHECKPOINT_DIR}/model.safetensors')
        print(f'  ✔ New best val loss: {best_val_loss:.4f} — model.safetensors saved.')
    else:
        early_stop_counter += 1
        print(f'  No improvement ({early_stop_counter}/{PATIENCE})')
        if early_stop_counter >= PATIENCE:
            print('Early stopping triggered.')
            break

    # ── Save resume checkpoint to Drive every epoch ──
    checkpoint.save_checkpoint(
        model, optimizer, scheduler, epoch,
        best_val_loss, early_stop_counter,
        train_losses, val_losses,
        RESUME_CHECKPOINT,
    )

model.load_state_dict(best_model_wts)
print(f'\nTraining complete. Best Val Loss: {best_val_loss:.4f}')

## Loss Curves

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train Dice Loss')
plt.plot(val_losses,   label='Val Dice Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MBEN + Attention U-Net Training & Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()

## Quick Inference Check

In [ ]:
FEATURE_ORDER = ('prnu', 'illumination', 'frequency')
active_features = [f for f in FEATURE_ORDER if f in set(FEATURES)]

model.eval()
batch = next(iter(val_loader))
*feat_tensors, fused, masks = batch
feature_dict = {
    feat: feat_tensors[i].to(device)
    for i, feat in enumerate(active_features)
}
fused = fused.to(device)

with torch.no_grad():
    preds = torch.sigmoid(model(feature_dict, fused)).cpu().numpy()

n_show  = min(4, fused.size(0))
n_plots = len(active_features) + 2  # features + ground truth + prediction
fig, axes = plt.subplots(n_plots, n_show, figsize=(14, 3.5 * n_plots))
if n_plots == 1:
    axes = axes[None, :]  # ensure 2-D indexing

for i in range(n_show):
    for row, feat in enumerate(active_features):
        axes[row, i].imshow(feat_tensors[row][i, 0].numpy(), cmap='gray')
        axes[row, i].set_title(feat.capitalize())
    axes[-2, i].imshow(masks[i, 0].numpy(), cmap='gray')
    axes[-2, i].set_title('Ground Truth')
    axes[-1, i].imshow(preds[i, 0],         cmap='gray')
    axes[-1, i].set_title('Prediction')

for ax in axes.flatten():
    ax.axis('off')
plt.tight_layout()
plt.savefig('inference_sample.png', dpi=150)
plt.show()

## Save Final Model

### Save to Hugging Face Repository

In [ ]:
from huggingface_hub import HfApi, login

login(token="write-token")

api = HfApi()

# Create the repo first (only needed once)
username = 'hf-username'
api.create_repo(repo_id=f"{username}/mben-unetpp", exist_ok=True)

# Upload the weights
api.upload_file(
    path_or_fileobj=f"{CHECKPOINT_DIR}/model.safetensors",
    path_in_repo='model.safetensors',
    repo_id=f"{username}/mben-unetpp",
)

In [ ]:
import json

config = {
    "model_type": "MBENUNetPlusPlus",
    "mben_out_ch": MBEN_OUT_CHANNELS,
    "features": sorted(FEATURES),
    "encoder": "efficientnet-b4",
    "encoder_weights": "imagenet",
    "in_channels": 3,
    "classes": 1,
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

api.upload_file(
    path_or_fileobj="config.json",
    path_in_repo="config.json",
    repo_id=f"{username}/mben-unetpp",
)
